<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/WithoutKD_AL9EE_B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip "/content/Apple.zip"

Streaming output truncated to the last 5000 lines.
  inflating: Apple/Scab/Scab (1368).jpg  
  inflating: Apple/Scab/Scab (1369).jpg  
  inflating: Apple/Scab/Scab (137).jpg  
  inflating: Apple/Scab/Scab (1370).jpg  
  inflating: Apple/Scab/Scab (1371).jpg  
  inflating: Apple/Scab/Scab (1372).jpg  
  inflating: Apple/Scab/Scab (1373).jpg  
  inflating: Apple/Scab/Scab (1374).jpg  
  inflating: Apple/Scab/Scab (1375).jpg  
  inflating: Apple/Scab/Scab (1376).jpg  
  inflating: Apple/Scab/Scab (1377).jpg  
  inflating: Apple/Scab/Scab (1378).jpg  
  inflating: Apple/Scab/Scab (1379).jpg  
  inflating: Apple/Scab/Scab (138).jpg  
  inflating: Apple/Scab/Scab (1380).jpg  
  inflating: Apple/Scab/Scab (1381).jpg  
  inflating: Apple/Scab/Scab (1382).jpg  
  inflating: Apple/Scab/Scab (1383).jpg  
  inflating: Apple/Scab/Scab (1384).jpg  
  inflating: Apple/Scab/Scab (1385).jpg  
  inflating: Apple/Scab/Scab (1386).jpg  
  inflating: Apple/Scab/Scab (1387).jpg  
  inflating: Apple/Scab/Sca

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [ ]:
!pip install split-folders

In [ ]:
import splitfolders

splitfolders.ratio(
    "/content/Apple",       # <-- your dataset folder
    output="/content/Apple_split",
    seed=42,
    ratio=(0.7, 0.15, 0.15) # train/val/test
)

Copying files: 11258 files [00:06, 1791.81 files/s]


In [ ]:
# =======================
# CONFIG
# =======================
DATA_DIR = "/content/Apple_split"  # <-- Change path here!
IMAGE_SIZE = (380, 380)
BATCH_SIZE = 16
NUM_CLASSES = 3
EPOCHS = 10

In [ ]:
# =======================
# DATA PIPELINE
# =======================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR + "/train", target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical'
)

val_gen = val_test_datagen.flow_from_directory(
    DATA_DIR + "/val", target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical'
)

test_gen = val_test_datagen.flow_from_directory(
    DATA_DIR + "/test", target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False
)

Found 7879 images belonging to 3 classes.
Found 1687 images belonging to 3 classes.
Found 1692 images belonging to 3 classes.


In [ ]:
# =======================
# MODEL - EfficientNetB4
# =======================
base_model = EfficientNetB4(
    include_top=False, weights='imagenet',
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
)

base_model.trainable = False  # Freeze for transfer learning

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 380, 380,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 380, 380,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 380, 380,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 380, 380,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 381, 381,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 190, 190,  │      1,296 │ stem_conv_pad[0]… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 190, 190,  │        192 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 190, 190,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 190, 190,  │        432 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 190, 190,  │        192 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 190, 190,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 48)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 48)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 12)  │        588 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 48)  │        624 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 190, 190,  │          0 │ block1a_activati… │
│ (Multiply)          │ 48)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 190, 190,  │      1,152 │ block1a_se_excit

 Total params: 17,679,202 (67.44 MB)

 Trainable params: 5,379 (21.01 KB)

 Non-trainable params: 17,673,823 (67.42 MB)

In [ ]:
# =======================
# CALLBACKS
# =======================
callbacks = [
    EarlyStopping(patience=6, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.2, patience=3, verbose=1),
    ModelCheckpoint("best_effnet_b4.h5", save_best_only=True)
]

In [ ]:
# =======================
# TRAIN
# =======================
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

# Unfreeze for fine-tuning
base_model.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

history_finetune = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 630ms/step - accuracy: 0.4617 - loss: 1.0709

493/493 ━━━━━━━━━━━━━━━━━━━━ 401s 703ms/step - accuracy: 0.4617 - loss: 1.0709 - val_accuracy: 0.4807 - val_loss: 1.0535 - learning_rate: 1.0000e-04
Epoch 2/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 563ms/step - accuracy: 0.4628 - loss: 1.0704

493/493 ━━━━━━━━━━━━━━━━━━━━ 293s 593ms/step - accuracy: 0.4628 - loss: 1.0704 - val_accuracy: 0.4807 - val_loss: 1.0517 - learning_rate: 1.0000e-04
Epoch 3/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 291s 591ms/step - accuracy: 0.4711 - loss: 1.0672 - val_accuracy: 0.4807 - val_loss: 1.0527 - learning_rate: 1.0000e-04
Epoch 4/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 564ms/step - accuracy: 0.4832 - loss: 1.0563

493/493 ━━━━━━━━━━━━━━━━━━━━ 293s 594ms/step - accuracy: 0.4832 - loss: 1.0563 - val_accuracy: 0.4807 - val_loss: 1.0515 - learning_rate: 1.0000e-04
Epoch 5/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 294s 597ms/step - accuracy: 0.4820 - loss: 1.0562 - val_accuracy: 0.4807 - val_loss: 1.0516 - learning_rate: 1.0000e-04
Epoch 6/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 293s 593ms/step - accuracy: 0.4684 - loss: 1.0719 - val_accuracy: 0.4807 - val_loss: 1.0545 - learning_rate: 1.0000e-04
Epoch 7/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 566ms/step - accuracy: 0.4495 - loss: 1.0743
Epoch 7: ReduceLROnPlateau reducing learning rate to 1.9999999494757503e-05.
493/493 ━━━━━━━━━━━━━━━━━━━━ 293s 594ms/step - accuracy: 0.4495 - loss: 1.0743 - val_accuracy: 0.4807 - val_loss: 1.0541 - learning_rate: 1.0000e-04
Epoch 8/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 571ms/step - accuracy: 0.4812 - loss: 1.0642

...

493/493 ━━━━━━━━━━━━━━━━━━━━ 371s 753ms/step - accuracy: 0.9528 - loss: 0.1440 - val_accuracy: 0.9846 - val_loss: 0.0615 - learning_rate: 1.0000e-05
Epoch 5/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 715ms/step - accuracy: 0.9717 - loss: 0.0932

493/493 ━━━━━━━━━━━━━━━━━━━━ 368s 747ms/step - accuracy: 0.9717 - loss: 0.0932 - val_accuracy: 0.9846 - val_loss: 0.0573 - learning_rate: 1.0000e-05
Epoch 6/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 715ms/step - accuracy: 0.9695 - loss: 0.0921

493/493 ━━━━━━━━━━━━━━━━━━━━ 368s 746ms/step - accuracy: 0.9696 - loss: 0.0921 - val_accuracy: 0.9870 - val_loss: 0.0460 - learning_rate: 1.0000e-05
Epoch 7/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 366s 742ms/step - accuracy: 0.9773 - loss: 0.0746 - val_accuracy: 0.9822 - val_loss: 0.0525 - learning_rate: 1.0000e-05
Epoch 8/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 706ms/step - accuracy: 0.9835 - loss: 0.0567

493/493 ━━━━━━━━━━━━━━━━━━━━ 364s 738ms/step - accuracy: 0.9835 - loss: 0.0567 - val_accuracy: 0.9887 - val_loss: 0.0411 - learning_rate: 1.0000e-05
Epoch 9/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 710ms/step - accuracy: 0.9848 - loss: 0.0454

493/493 ━━━━━━━━━━━━━━━━━━━━ 366s 742ms/step - accuracy: 0.9848 - loss: 0.0454 - val_accuracy: 0.9887 - val_loss: 0.0369 - learning_rate: 1.0000e-05
Epoch 10/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 365s 741ms/step - accuracy: 0.9870 - loss: 0.0399 - val_accuracy: 0.9893 - val_loss: 0.0375 - learning_rate: 1.0000e-05


In [ ]:
# ======================= # TEST EVALUATION # =======================
test_loss, test_acc = model.evaluate(test_gen)
print(f"Test Accuracy: {test_acc:.4f}")

106/106 ━━━━━━━━━━━━━━━━━━━━ 34s 319ms/step - accuracy: 0.9848 - loss: 0.0390
Test Accuracy: 0.9870


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

# Get predictions
y_true = test_gen.classes
y_pred_prob = model.predict(test_gen)
y_pred = np.argmax(y_pred_prob, axis=1)

# Class labels mapping
class_labels = list(test_gen.class_indices.keys())

# Classification report (Precision, Recall, F1-score & Accuracy included)
report = classification_report(y_true, y_pred, target_names=class_labels, digits=4)
print("Classification Report:\n")
print(report)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

# Cohen's Kappa
kappa = cohen_kappa_score(y_true, y_pred)
print(f"\nCohen's Kappa: {kappa:.4f}")

106/106 ━━━━━━━━━━━━━━━━━━━━ 40s 264ms/step
Classification Report:

              precision    recall  f1-score   support

      Health     0.9978    0.9871    0.9924       465
        Rust     0.9901    0.9686    0.9792       414
        Scab     0.9794    0.9963    0.9878       813

    accuracy                         0.9870      1692
   macro avg     0.9891    0.9840    0.9865      1692
weighted avg     0.9871    0.9870    0.9870      1692

Confusion Matrix:
[[459   1   5]
 [  1 401  12]
 [  0   3 810]]

Cohen's Kappa: 0.9794
